In [1]:
import pandas as pd
import numpy as np

In [31]:
data = {
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006,
                1007, 1008, 1009, 1010, 1011, 1012],
    "Product": ["Laptop", "Mouse", "Keyboard", "Laptop",
                "Monitor", "Mouse", "Keyboard", "Desk",
                "Laptop", "Monitor", "Mouse", "Desk"],
    "Category": ["Electronics", "Electronics", "Electronics",
                 "Electronics", "Electronics", "Electronics",
                 "Electronics", "Furniture", "Electronics",
                 "Electronics", "Electronics", "Furniture"],
    "Quantity": [2, 5, 3, 1, np.nan, 4, 2, 1, 3, 2, np.nan, 2],
    "UnitPrice": [800, 20, 50, 800, 250, 20, 50, 300,
                  800, 250, 20, 300],
    "Region": ["North", "South", "North", "East", "West",
               "South", "North", "East", "West", "West",
               "South", "East"],
    "OrderDate": ["2026-09-01", "2026-09-02", "2026-09-03",
                  "2026-09-04", "2026-09-05", "2026-09-06",
                  "2026-09-07", "2026-09-08", "2026-09-09",
                  "2026-09-10", "2026-09-11", "2026-09-12"]
}

df = pd.DataFrame(data)

In [19]:
df.head(12)

,OrderID,Product,Category,Quantity,UnitPrice,Region,OrderDate
0,1001,Laptop,Electronics,2.0,800,North,2026-09-01
1,1002,Mouse,Electronics,5.0,20,South,2026-09-02
2,1003,Keyboard,Electronics,3.0,50,North,2026-09-03
3,1004,Laptop,Electronics,1.0,800,East,2026-09-04
4,1005,Monitor,Electronics,NaN,250,West,2026-09-05
5,1006,Mouse,Electronics,4.0,20,South,2026-09-06
6,1007,Keyboard,Electronics,2.0,50,North,2026-09-07
7,1008,Desk,Furniture,1.0,300,East,2026-09-08
8,1009,Laptop,Electronics,3.0,800,West,2026-09-09
9,1010,Monitor,Electronics,2.0,250,West,2026-09-10


In [46]:
df.info()
df.shape
df.dtypes
df["OrderDate"]=pd.to_datetime(df["OrderDate"])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   OrderID    12 non-null     int64         
 1   Product    12 non-null     object        
 2   Category   12 non-null     object        
 3   Quantity   10 non-null     float64       
 4   UnitPrice  12 non-null     int64         
 5   Region     12 non-null     object        
 6   OrderDate  12 non-null     datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 804.0+ bytes


In [36]:
df.isna().sum()


OrderID      0
Product      0
Category     0
Quantity     2
UnitPrice    0
Region       0
OrderDate    0
dtype: int64

In [14]:
df.duplicated()

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
dtype: bool

In [28]:
df.groupby(["Product"])["Quantity"].agg(
    ["count", "mean", "median"]
)
df.groupby(["Product","Region"])["Quantity"].agg(
    ["count", "mean", "median", "sum"]
)

count  mean  median  sum
Product  Region                          
Desk     East        2   1.5     1.5  3.0
Keyboard North       2   2.5     2.5  5.0
Laptop   East        1   1.0     1.0  1.0
         North       1   2.0     2.0  2.0
         West        1   3.0     3.0  3.0
Monitor  West        1   2.0     2.0  2.0
Mouse    South       2   4.5     4.5  9.0

In [45]:
df_no_missing=df.copy()
df_no_missing["Quantity_imputed"] = df_no_missing["Quantity"].isna()
df_no_missing["Quantity"] = df.groupby(
    ["Product", "Region"]
)["Quantity"].transform(
    lambda x: x.fillna(np.ceil(x.median()))
)
df_no_missing["Quantity"]=df_no_missing["Quantity"].astype(int)
print(df_no_missing)

    OrderID   Product     Category  Quantity  UnitPrice Region  OrderDate  \
0      1001    Laptop  Electronics         2        800  North 2026-09-01   
1      1002     Mouse  Electronics         5         20  South 2026-09-02   
2      1003  Keyboard  Electronics         3         50  North 2026-09-03   
3      1004    Laptop  Electronics         1        800   East 2026-09-04   
4      1005   Monitor  Electronics         2        250   West 2026-09-05   
5      1006     Mouse  Electronics         4         20  South 2026-09-06   
6      1007  Keyboard  Electronics         2         50  North 2026-09-07   
7      1008      Desk    Furniture         1        300   East 2026-09-08   
8      1009    Laptop  Electronics         3        800   West 2026-09-09   
9      1010   Monitor  Electronics         2        250   West 2026-09-10   
10     1011     Mouse  Electronics         5         20  South 2026-09-11   
11     1012      Desk    Furniture         2        300   East 2026-09-12   

In [47]:
df_no_missing["Quantity_imputed"]

0     False
1     False
2     False
3     False
4      True
5     False
6     False
7     False
8     False
9     False
10     True
11    False
Name: Quantity_imputed, dtype: bool

In [49]:
df_no_missing["Revenue"]=df_no_missing["Quantity"]*df_no_missing["UnitPrice"]
df_no_missing.head()

,OrderID,Product,Category,Quantity,UnitPrice,Region,OrderDate,Quantity_imputed,Revenue
0,1001,Laptop,Electronics,2,800,North,2026-09-01,False,1600
1,1002,Mouse,Electronics,5,20,South,2026-09-02,False,100
2,1003,Keyboard,Electronics,3,50,North,2026-09-03,False,150
3,1004,Laptop,Electronics,1,800,East,2026-09-04,False,800
4,1005,Monitor,Electronics,2,250,West,2026-09-05,True,500


In [55]:
total=df_no_missing["Revenue"].sum()
av_rev_or=round(df_no_missing["Revenue"].sum()/(df_no_missing["OrderID"].count()),2)
print(total, av_rev_or)

7230 602.5


In [61]:
regional_summary=df_no_missing.groupby(["Region"])["Revenue"].agg(["sum", "mean","count", "median" ]).sort_values("sum", ascending=False)
regional_summary

,sum,mean,count,median
Region,,,,
West,3400,1133.333333,3,500.0
North,1850,616.666667,3,150.0
East,1700,566.666667,3,600.0
South,280,93.333333,3,100.0


In [66]:
product_summary=df_no_missing.groupby(["Product"]).agg(
        total_quantity=("Quantity", "sum"),
        total_revenue=("Revenue", "sum"),
        average_revenue=("Revenue", "mean"),
        number_of_orders=("OrderID", "count")
    ).sort_values("total_revenue", ascending=False)
product_summary

,total_quantity,total_revenue,average_revenue,number_of_orders
Product,,,,
Laptop,6,4800,1600.000000,3
Monitor,4,1000,500.000000,2
Desk,3,900,450.000000,2
Mouse,14,280,93.333333,3
Keyboard,5,250,125.000000,2
